<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/GRUPO1_PedroAbrah%C3%A3o_pmc_varejo_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sidrapy

from datetime import date
import pandas as pd
import sidrapy

# Busca toda a série histórica (de jan/2000 até dez/2026) apenas para Volume de vendas
df_bruto = sidrapy.get_table(
    table_code="8880",
    territorial_level="1",
    ibge_territorial_code="all",
    classifications={"11046": "56734"},
    period="200001-202612"
)

# Descobre dinamicamente qual coluna é Mês (Código) e qual é Variável (Código)
cabecalho = df_bruto.iloc[0]
col_mes = cabecalho[cabecalho == "Mês (Código)"].index[0]
col_var = cabecalho[cabecalho == "Variável (Código)"].index[0]

df_dados = df_bruto.iloc[1:].copy()

mapa_vars = {
    "7169": "Indice_Volume",
    "7170": "Indice_Ajuste_Sazonal",
    "11708": "Var_Mensal_SA",
    "11709": "Var_Interanual",
    "11710": "Var_Acum_Ano",
    "11711": "Var_Acum_12M"
}

tabela_base = pd.DataFrame()
tabela_base["periodo"] = pd.to_datetime(df_dados[col_mes], format="%Y%m")
tabela_base["indicador"] = df_dados[col_var].map(mapa_vars)
tabela_base["valor"] = pd.to_numeric(df_dados["V"], errors="coerce")

tabela_base = tabela_base.dropna(subset=["indicador", "valor"]).sort_values(["indicador", "periodo"]).reset_index(drop=True)

hoje = date.today().isoformat()
tabela_base.to_csv(f"tabela_base_{hoje}.csv", index=False)

inicio = tabela_base["periodo"].min().strftime("%Y-%m")
fim = tabela_base["periodo"].max().strftime("%Y-%m")
print(f"Série completa carregada: de {inicio} até {fim}")
tabela_base.head(10)

Série completa carregada: de 2000-01 até 2026-09


,periodo,indicador,valor
0,2000-01-01,Indice_Ajuste_Sazonal,50.14033
1,2000-02-01,Indice_Ajuste_Sazonal,50.65798
2,2000-03-01,Indice_Ajuste_Sazonal,50.97725
3,2000-04-01,Indice_Ajuste_Sazonal,50.94511
4,2000-05-01,Indice_Ajuste_Sazonal,51.28520
5,2000-06-01,Indice_Ajuste_Sazonal,51.12965
6,2000-07-01,Indice_Ajuste_Sazonal,51.32649
7,2000-08-01,Indice_Ajuste_Sazonal,50.78226
8,2000-09-01,Indice_Ajuste_Sazonal,50.98035
9,2000-10-01,Indice_Ajuste_Sazonal,50.21967


In [2]:
# Função que pega o valor anterior e divide o atual pelo anterior (variação %)
def calc_var_anterior(serie, defasagem=12):
    anterior = serie.shift(defasagem)
    return ((serie / anterior) - 1) * 100


# Tabela 1: Compara os 12 meses de um ano escolhido contra os 12 meses do ano anterior
def tabela_12_meses_por_ano(df):
    df_piv = df.pivot(index="periodo", columns="indicador", values="valor").sort_index()

    anos_disp = sorted(df_piv.index.year.unique().tolist())
    print(f"Anos disponíveis na base: {anos_disp[0]} até {anos_disp[-1]}")
    ano_escolhido = int(input("Digite o ano para ver os 12 meses vs 12 meses anteriores (ex: 2024): ").strip())

    if ano_escolhido not in anos_disp:
        print("Ano não encontrado na base.")
        return None

    tab = pd.DataFrame(index=df_piv.index)
    tab[f"Indice_{ano_escolhido}"] = df_piv["Indice_Volume"]
    tab[f"Indice_{ano_escolhido - 1}"] = df_piv["Indice_Volume"].shift(12)
    tab["Var_12M_Calc (%)"] = calc_var_anterior(df_piv["Indice_Volume"], defasagem=12)

    if "Var_Interanual" in df_piv.columns:
        tab["Var_IBGE (%)"] = df_piv["Var_Interanual"]
    if "Var_Acum_12M" in df_piv.columns:
        tab["Acum_12M (%)"] = df_piv["Var_Acum_12M"]

    # Filtra apenas os 12 meses do ano que você escolheu
    recorte = tab[tab.index.year == ano_escolhido].copy()
    recorte.index = recorte.index.strftime("%b/%Y")
    recorte.index.name = "Mês"
    return recorte.round(2)


# Tabela 2: Compara os resultados de Dezembro dos anos que você escolher
def tabela_dezembros_escolha(df):
    df_piv = df.pivot(index="periodo", columns="indicador", values="valor").sort_index()

    # Filtra apenas Dezembro de todos os anos primeiro para calcular a variação contra o Dezembro anterior
    df_dez = df_piv[df_piv.index.month == 12].copy()

    tab_dez = pd.DataFrame(index=df_dez.index)
    tab_dez["Ano"] = df_dez.index.year
    tab_dez["Indice_Dez"] = df_dez["Indice_Volume"]
    tab_dez["Indice_Dez_Anterior"] = df_dez["Indice_Volume"].shift(1)
    tab_dez["Var_Dez_vs_Dez_Ant (%)"] = calc_var_anterior(df_dez["Indice_Volume"], defasagem=1)

    if "Var_Acum_Ano" in df_dez.columns:
        tab_dez["Fechamento_Ano_IBGE (%)"] = df_dez["Var_Acum_Ano"]

    entrada = input("Quais anos de Dezembro quer comparar? (ex: 2015-2025 ou 2013,2020,2024): ").strip()

    if "-" in entrada:
        ini, fim = map(int, entrada.split("-"))
        recorte = tab_dez[(tab_dez["Ano"] >= ini) & (tab_dez["Ano"] <= fim)].copy()
    else:
        lista_anos = [int(a.strip()) for a in entrada.split(",")]
        recorte = tab_dez[tab_dez["Ano"].isin(lista_anos)].copy()

    recorte = recorte.drop(columns=["Ano"])
    recorte.index = recorte.index.strftime("Dez/%Y")
    recorte.index.name = "Referência"
    return recorte.round(2)

In [ ]:
tab_12m = tabela_12_meses_por_ano(tabela_base)
if tab_12m is not None:
    print("\n=== TABELA 1: 12 MESES DO ANO ESCOLHIDO VS ANO ANTERIOR ===")
    display(tab_12m)
    tab_12m.to_csv("tabela_12_meses_escolhida.csv")

print("-" * 60)

tab_dez = tabela_dezembros_escolha(tabela_base)
if tab_dez is not None:
    print("\n=== TABELA 2: RESULTADOS DE DEZEMBRO DOS ANOS ESCOLHIDOS ===")
    display(tab_dez)
    tab_dez.to_csv("tabela_dezembros_escolhida.csv")

Anos disponíveis na base: 2000 até 2026
